# Runnable Config

In [3]:
from langchain_core.runnables import RunnableConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# The State only holds the actual conversation payload
class AgentState(TypedDict):
    messages: list
    issue_resolved: bool

def dynamic_support_node(state: AgentState, config: RunnableConfig):
    # 1. Extract business logic parameters from the 'configurable' key
    user_id = config.get("configurable", {}).get("user_id", "anonymous")
    subscription_tier = config.get("configurable", {}).get("tier", "free")

    # 2. Dynamically instantiate the LLM based on the Config!
    # The LLM never sees the "tier" variable, but our architecture uses it.
    if subscription_tier == "premium":
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")
        system_prompt = f"You are a white-glove support agent for our VIP user {user_id}."
    else:
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
        system_prompt = "You are a concise support agent. Keep answers brief."

    # 3. Access other built-in config features (like checking recursion)
    # If the graph has looped 10 times, we can force it to stop
    current_step = config.get("configurable", {}).get("step", 0)

    # ... generate the response ...
    messages_to_pass = [{"role": "system", "content": system_prompt}] + state["messages"]
    response = llm.invoke(messages_to_pass)

    return {"messages": [response]}

# Build the Graph
builder = StateGraph(AgentState)
builder.add_node("support", dynamic_support_node)
builder.add_edge(START, "support")
builder.add_edge("support", END)
app = builder.compile()

In [4]:
# The payload (State)
user_message = {"messages": [{"role": "user", "content": "My account is locked!"}]}

# The execution metadata (RunnableConfig)
execution_config = {
    # 'configurable' holds your custom variables
    "configurable": {
        "user_id": "usr_99827",
        "tier": "premium",
        "thread_id": "conv_123" # LangGraph's checkpointer uses this automatically!
    },
    # 'recursion_limit' prevents infinite loops (default is usually 25)
    "recursion_limit": 5,
    # 'tags' for monitoring in LangSmith
    "tags": ["billing_issue", "premium_support_queue"]
}

# Run the graph
result = app.invoke(user_message, config=execution_config)

print(result["messages"][-1].content)

Hello usr_99827,

I'm so sorry to hear you're locked out of your account. I understand how critical access is, and I'm making it my top priority to get this resolved for you immediately.

To securely locate and verify your account, could you please provide me with the primary email address you have on file?

Once I have that, I will personally handle the unlock process. This is usually a straightforward security measure, and we'll have you back in no time.

I'll be standing by for your reply.


In [5]:
# The payload (State)
user_message = {"messages": [{"role": "user", "content": "My account is locked!"}]}

# The execution metadata (RunnableConfig)
execution_config = {
    # 'configurable' holds your custom variables
    "configurable": {
        "user_id": "usr_99827",
        "tier": "free",
        "thread_id": "conv_123" # LangGraph's checkpointer uses this automatically!
    },
    # 'recursion_limit' prevents infinite loops (default is usually 25)
    "recursion_limit": 5,
    # 'tags' for monitoring in LangSmith
    "tags": ["billing_issue", "premium_support_queue"]
}

# Run the graph
result = app.invoke(user_message, config=execution_config)

print(result["messages"][-1].content)

Contact support for assistance. This is usually for security reasons.
